# ⏩ 使用 GitHub 模型的顺序代理工作流（Python）

## 📋 高级顺序处理教程

本笔记本演示了使用 Microsoft Agent Framework 的<strong>顺序工作流模式</strong>。您将学习如何构建复杂的多步骤处理管道，其中代理按特定顺序执行，并在各阶段之间传递数据和上下文。

## 🎯 学习目标

### 🔄 <strong>顺序处理模式</strong>
- <strong>线性工作流设计</strong>：创建逐步处理管道
- <strong>数据流管理</strong>：在顺序代理之间传递信息
- <strong>阶段门控处理</strong>：实现检查点和验证阶段
- <strong>进度跟踪</strong>：监控工作流执行和中间结果

### 🏗️ <strong>企业管道架构</strong>
- <strong>业务流程建模</strong>：将实际业务流程映射到代理工作流
- <strong>质量保证</strong>：多阶段验证和审核流程
- <strong>文档处理</strong>：顺序文档分析和转换
- <strong>内容制作</strong>：带有审核和批准阶段的编辑工作流

### 📊 <strong>高级工作流特性</strong>
- <strong>上下文保持</strong>：维护工作流各阶段的状态
- <strong>错误传播</strong>：处理顺序处理中的失败
- <strong>性能优化</strong>：高效的顺序执行模式
- <strong>审计追踪</strong>：完整跟踪顺序操作

## ⚙️ 先决条件与设置

### 📦 <strong>依赖项</strong>
```bash
pip install agent-framework-core -U
```

### 🔑 <strong>配置</strong>
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

## 🏢 <strong>企业顺序工作流用例</strong>

### 📝 <strong>文档处理管道</strong>
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 <strong>质量保证工作流</strong> 
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 <strong>内容制作管道</strong>
```
Research → Writing → Editing → Review → Publishing
```

### 💼 <strong>业务流程自动化</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 <strong>顺序工作流设计原则</strong>

- **🔗 线性进展**：每个阶段依赖于前一阶段的输出
- **📋 状态管理**：跨所有阶段保持上下文和数据
- **🛡️ 错误处理**：优雅地管理任一阶段的失败
- **📊 进度监控**：跟踪每个阶段的完成情况和性能
- **🔄 阶段可重用性**：设计可重用的工作流组件

让我们构建复杂的顺序处理工作流吧！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = await provider.create_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = await provider.create_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = await provider.create_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
